<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_2_Your_First_Claude_API_Call.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 — Your First Claude API Call
### AI Architect Mastery Program · Duration ~2 hours · Level: Beginner

Today you make a real program talk to Claude. You will **send a prompt, read the reply, and measure what it cost in tokens** — the single most important loop in everything you'll build later.

**What you'll be able to do by the end**
- **Explain** what an API is and what one Claude call actually contains.
- **Demonstrate** a working call: prompt in, answer out.
- **Implement** a Python script from scratch that calls the live Claude API.
- **Inspect** token usage and estimate cost.
- **Architect** the place this call sits inside a real production system.

**How each section is structured**

| Block | What it gives you |
|---|---|
| The simple questions | What / why / how / when |
| Real code you run | The idea, live against Claude |
| Remember this | One interview-ready line |
| Don't mix these up | Common beginner traps |
| Interview + Quick check | Prove you got it |

# Foundations — the words you need first

A few plain definitions before we touch code. Read once; you'll see them in action below.

- **LLM (Large Language Model):** a program that predicts the next piece of text. Claude is an LLM.
- **Prompt:** the text you send in (your question or instruction).
- **Response:** the text Claude sends back.
- **API (Application Programming Interface):** a doorway that lets *your code* talk to Claude's servers. You send a request over the internet; you get a reply.
- **Token:** a small chunk of text (roughly three-quarters of a word). Claude reads and writes in tokens, and **you pay per token**.
- **SDK (Software Development Kit):** a ready-made library so you don't handle the raw internet plumbing. We use Anthropic's `anthropic` Python SDK.

Picture of the whole thing:

```
You (your code)  --prompt-->  [ Claude API ]  --response-->  You (your code)
```


# Section 1 — What is a Claude API call?

**What is it?** One API call is one round trip: your code sends a prompt to Claude over the internet and gets back text. That's it.

**Why does it matter?** This single call is the atom of every AI product — chatbots, copilots, agents, RAG systems. Master this and everything else is just *more of these calls, arranged cleverly*.

**How does it work? (step by step)**
1. Your code builds a request: which **model**, the **messages** (your prompt), and a **max_tokens** limit.
2. The SDK sends it to Claude's servers with your API key.
3. Claude generates a reply.
4. You read the text out of the response object.

**When would I use it?** Any time you want a program (not a human in a chat window) to use Claude — automation, backends, batch jobs, agents.

**When NOT to use it?** For quick one-off questions, the Claude.ai chat app is easier and cheaper. Use the API when you need *code* in the loop.

> **Accuracy note:** an API call is **stateless** — Claude does not remember your last call. "Memory" in a chatbot is just your code re-sending past messages each time. Interviewers love this one.


**Remember this**
> A Claude API call is one stateless round trip: prompt in, text out. Memory is something *your code* adds, not the API.

**Don't mix these up**
- ❌ "The API remembers our conversation." → ✅ Each call is independent; you resend history yourself.
- ❌ "Claude.ai and the API are the same product." → ✅ Same model, different access: a chat app vs. a pay-per-token programmatic doorway.
- ❌ "max_tokens is the length of my prompt." → ✅ It caps the **reply** length, not the input.

**Interview question**
*Q: Is the Claude API stateful or stateless, and why does it matter?*
A: Stateless — each call stands alone. It matters because *you* must resend prior messages to create a conversation, and that resent history is tokens you pay for every turn.

**Quick check**
1. What three things does a basic call need? *(model, messages, max_tokens)*
2. Where does conversation memory actually live? *(in your code, by resending messages)*


# Section 2 — Get set up (install + key)

Two small steps. Run them once per session.

**Step A — install the SDK.** This downloads Anthropic's Python library.

**Step B — give your code a key.** The API key proves who you are and bills you. **Never paste a real key into a shared notebook.** In Google Colab, store it as a *Secret* named `MY_API_KEY` (the key panel on the left), and read it in code.

> Getting a key: go to platform.claude.com then Console then Manage then API Keys then Create Key. Add a few dollars of credit to start.


In [1]:
# Step A — install the Anthropic SDK (run once)
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 4.4 MB/s eta 0:00:00


In [2]:
# Step B — load your key from a Colab Secret named MY_API_KEY, then make a client
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")  # never hardcode the key

import anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-4-5-20251001"     # fast + cheap: the right default for learning
print("Ready")

Ready


# Section 3 — Your first call (prompt in, answer out)

This is the whole point of today. Three required pieces:
- **model** — which Claude to use (`MODEL`, our Haiku).
- **max_tokens** — the longest reply you'll allow.
- **messages** — a list of turns; each turn has a `role` (`"user"`) and `content` (your text).

The answer comes back at `reply.content[0].text`.


In [3]:
reply = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "Say hello and explain what an API is in one sentence."}
    ],
)

print(reply.content[0].text)

Hello! An API (Application Programming Interface) is a set of rules and tools that allows different software programs to communicate with each other and share data or functionality.


**Why `content[0].text`?** Claude's reply is a *list of blocks* (usually one text block). `content[0]` is the first block; `.text` is its words. Later, with tools, you'll see more than one block — same shape.

**Try this:** change the prompt to ask for the answer "as if explaining to a 10-year-old" and run again.

**Remember this**
> The answer lives at `reply.content[0].text` — `content` is a list of blocks, not a plain string.


# Section 4 — Read the whole response object

The reply is more than text. Two fields you'll use constantly:
- **`reply.stop_reason`** — *why* Claude stopped. `"end_turn"` = finished normally. `"max_tokens"` = it ran out of room (answer cut off). `"tool_use"` = it wants to call a tool (you'll meet this in the agents session).
- **`reply.model`** — which model actually answered.

Checking `stop_reason` is how your *code* knows what to do next.


In [5]:
print("text        :", reply.content[0].text[:80], "...")
print("stop_reason :", reply.stop_reason)   # 'end_turn' means it finished cleanly
print("model       :", reply.model)
print("block type  :", reply.content[0].type)

text        : Hello! An API (Application Programming Interface) is a set of rules and tools th ...
stop_reason : end_turn
model       : claude-haiku-4-5-20251001
block type  : text


# Section 5 — Inspect token usage (and cost)

Every reply carries a **`usage`** object:
- **`usage.input_tokens`** — tokens in your prompt (what Claude read).
- **`usage.output_tokens`** — tokens in the reply (what Claude wrote).

**You pay for both.** Pricing is *per million tokens*, and output usually costs more than input. So the two levers on your bill are: keep prompts lean, and keep `max_tokens` only as large as you need.

> **Why architects care:** at scale, tokens *are* the cloud bill. A chatbot that resends a long history every turn can quietly 10x its cost.


In [6]:
print("input tokens :", reply.usage.input_tokens)    # what Claude read
print("output tokens:", reply.usage.output_tokens)   # what Claude wrote
print("total tokens :", reply.usage.input_tokens + reply.usage.output_tokens)

input tokens : 19
output tokens: 35
total tokens : 54


In [7]:
# Estimate the cost of this one call (plug in current Haiku prices from platform.claude.com).
# Prices are per 1,000,000 tokens. Check the live pricing page before quoting real numbers.
PRICE_IN_PER_M  = 1.00   # <-- example placeholder $/1M input tokens
PRICE_OUT_PER_M = 5.00   # <-- example placeholder $/1M output tokens

cost = (reply.usage.input_tokens  / 1_000_000) * PRICE_IN_PER_M \
     + (reply.usage.output_tokens / 1_000_000) * PRICE_OUT_PER_M
print(f"Estimated cost of this call: ${cost:.8f}")

Estimated cost of this call: $0.00019400


**Remember this**
> You pay for input *and* output tokens, and output is usually pricier — short prompts and tight `max_tokens` are your cheapest wins.

**Try this:** ask for a 500-word essay, rerun, and watch `output_tokens` (and the cost) jump.


# Section 6 — Steer the answer: `system` and `temperature`

Two simple controls shape the reply:
- **`system`** — a separate instruction that sets Claude's role/rules for the whole call (e.g. *"You are a precise banking assistant. Answer in one line."*). It's the single best way to control tone and format.
- **`temperature`** — how *risky* the word choices are, from `0.0` (focused, repeatable) to `1.0` (creative, varied). Use low for facts/code, higher for brainstorming.

> **Accuracy note:** temperature does **not** change what Claude knows — only how adventurously it picks among likely words.


In [8]:
reply2 = client.messages.create(
    model=MODEL,
    max_tokens=120,
    system="You are a precise banking assistant. Answer in exactly one short sentence.",
    temperature=0.0,                         # focused and repeatable
    messages=[{"role": "user", "content": "What is an overdraft fee?"}],
)
print(reply2.content[0].text)
print("output tokens:", reply2.usage.output_tokens)

An overdraft fee is a charge your bank imposes when you spend more money than available in your account.
output tokens: 25


**Don't mix these up**
- ❌ "`system` is just another user message." → ✅ It's a separate, higher-priority instruction channel for rules/persona.
- ❌ "Higher temperature makes Claude smarter." → ✅ It only makes word choice more varied — sometimes worse for facts.

**Interview question**
*Q: A client wants identical answers every run for a compliance report. What do you set?*
A: `temperature=0.0` and a strict `system` prompt fixing the format; lock the model ID too.


# Lab — Build a "Claude in a script" tool from scratch

**Objective:** in one cell, write a small reusable function that sends any prompt, prints the answer, and reports tokens — your first real building block.

**Setup:** you already ran the install + key cells in Section 2. Reuse `client` and `MODEL`.


In [10]:
def ask_claude(prompt, max_tokens=300, system=None, temperature=0.7):
    """Send one prompt to Claude. Print the answer and the token usage."""
    message_params = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}],
    }
    if system is not None:
        message_params["system"] = system

    reply = client.messages.create(**message_params)
    text = reply.content[0].text
    print(text)
    print("-" * 40)
    print(f"in={reply.usage.input_tokens}  out={reply.usage.output_tokens}  stop={reply.stop_reason}")
    return reply

# Run it:
_ = ask_claude("Give me 3 startup ideas that use AI for small bakeries.")

# 3 AI Startup Ideas for Small Bakeries

## 1. **Smart Demand Forecasting**
An AI tool that predicts daily demand for specific items based on weather, local events, day of week, and historical sales. Helps bakeries minimize waste by baking the right quantities and suggests what to feature each day.

## 2. **Recipe Optimization & Scaling**
AI software that adjusts recipes based on ingredient costs, availability, and seasonal changes while maintaining quality. Also helps scale recipes up/down and calculates ingredient ratios automatically—saving time and reducing waste.

## 3. **Customer Preference & Loyalty Platform**
An AI-powered POS integration that learns customer preferences, suggests personalized recommendations, and automates targeted marketing (texts/emails). Helps small bakeries build repeat business without hiring a marketing team.

---

**Why these work:** They solve real problems (waste, consistency, customer retention), don't require massive infrastructure, and can integrat

**Expected result:** a short list of ideas, then a line like `in=20  out=140  stop=end_turn`.

**Try this (challenge):**
1. Call `ask_claude(..., system="Reply only in JSON with keys idea, why.")` and see the format change.
2. Set `temperature=0.0` and run the same prompt twice — notice how similar the two answers are.
3. Set `max_tokens=15` and watch `stop` become `max_tokens`.

**Cost & safety in one line each:** Haiku is cheap, but every call bills tokens — keep `max_tokens` sane; and never hardcode your key in a notebook you'll share.


# Architecture — where this one call lives in a real system

A single call rarely runs alone. In production it sits behind your own backend:

```
[ User / App ]
      |  request
      v
[ Your Backend API ]  --- holds the secret key, adds the system prompt,
      |                    logs tokens, retries on failure, enforces limits
      |  messages.create()
      v
[ Claude API ]
      |  response + usage
      v
[ Your Backend ]  --- reads stop_reason, stores cost, returns clean text
      |
      v
[ User / App ]
```

**Component responsibilities**
- **Your backend** keeps the API key secret (never ship it to the browser), attaches the `system` prompt, and records `usage` for billing.
- **Claude API** does the generation and returns `usage`.

**Failure points & fixes**
- Network/timeout: retry with backoff.
- `stop_reason == "max_tokens"`: raise the limit or ask for shorter output.
- Cost spikes: log `input_tokens`/`output_tokens` per call; alert on anomalies.

**Scaling / security / cost tradeoffs**
- **Security:** key lives server-side only; clients never see it.
- **Cost:** trim resent history, cap `max_tokens`, pick the smallest model that passes your quality bar (start at Haiku).
- **Reliability:** treat the call as a network operation — it can fail, so wrap it with retries in real code (kept out of beginner cells for clarity).


# Mini Project — "Support Reply Drafter" with a token budget

**Business use case:** a support team wants Claude to draft a first-response email to a customer complaint, in a fixed friendly tone, while the team tracks spend per draft.

**Architecture:** one `system` prompt fixes tone + format, one `messages.create` per complaint, log `usage` to estimate monthly cost.

**Build steps**
1. Write a `system` prompt: *"You are a calm, friendly support agent. Reply in under 80 words, apologize once, give one next step."*
2. Loop over a list of complaints and call Claude for each.
3. Print each draft plus its output tokens, and sum the tokens at the end.

**Definition of done:** every complaint gets a clean draft, and you can state the total tokens (and rough cost) for the batch.


In [ ]:
complaints = [
    "My order arrived broken and no one has replied to my email for 3 days.",
    "I was charged twice for the same subscription this month.",
    "The app keeps logging me out every few minutes.",
]

SYSTEM = ("You are a calm, friendly support agent. "
          "Reply in under 80 words, apologize once, and give one clear next step.")

total_out = 0
for i, c in enumerate(complaints, 1):
    reply = client.messages.create(
        model=MODEL, max_tokens=160, temperature=0.4,
        system=SYSTEM,
        messages=[{"role": "user", "content": f"Customer says: {c}"}],
    )
    print(f"--- Draft {i} ---")
    print(reply.content[0].text)
    print(f"[output tokens: {reply.usage.output_tokens}]\n")
    total_out += reply.usage.output_tokens

print("TOTAL output tokens for the batch:", total_out)

# Revision Suite

## Session Summary
You learned the atom of AI engineering: one **stateless** Claude API call. You installed the SDK, authenticated with a secret key, sent a prompt, read `content[0].text`, checked `stop_reason`, measured `usage` tokens (and cost), and steered output with `system` and `temperature`. Then you wrapped it in a reusable function and saw where it sits in a real backend.

## What You Learned Today
- You can **explain** what an API call is and why it's stateless.
- You can **implement** a working Claude call from scratch in Python.
- You can **read** `content[0].text`, `stop_reason`, and `usage`.
- You can **control** answers with `system` and `temperature`.
- You can **estimate** cost from token counts.


## AI Architect Cheat Sheet

**The core call**
```python
reply = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    system="optional rules/persona",
    temperature=0.0,            # 0 = focused, 1 = creative
    messages=[{"role": "user", "content": "your prompt"}],
)
reply.content[0].text          # the answer
reply.stop_reason              # 'end_turn' | 'max_tokens' | 'tool_use'
reply.usage.input_tokens       # billed
reply.usage.output_tokens      # billed (usually pricier)
```

**Model picker**

| Model | When |
|---|---|
| `claude-haiku-4-5-20251001` | default — fast, cheap, learning & high volume |
| `claude-sonnet-4-6` | deeper reasoning |
| `claude-opus-4-8` | hardest tasks (rare for beginners) |

**Cost levers:** shorter prompts, smaller `max_tokens`, trim resent history, smallest model that passes.


## 5-Minute Revision Guide
1. One call = prompt in, text out, **stateless**.
2. Needed: `model`, `max_tokens`, `messages`.
3. Answer = `reply.content[0].text` (content is a *list of blocks*).
4. `stop_reason`: `end_turn` good, `max_tokens` cut off, `tool_use` wants a tool.
5. `usage.input_tokens` + `usage.output_tokens` = your bill; output costs more.
6. `system` sets persona/format; `temperature` sets risk (0 = repeatable).
7. Keep the key server-side; never hardcode in shared files.

## Interview Preparation Notes
- *Stateless?* Yes — resend history yourself; that history is billed tokens.
- *Where's the answer?* `content[0].text`; content is a list (tools add blocks).
- *Empty/cut answer?* Check `stop_reason == "max_tokens"`, raise the limit.
- *Deterministic output?* `temperature=0.0` + strict `system` + pinned model.
- *(Architecture)* Where does the key live? Server-side only; clients call your backend, your backend calls Claude.
- *(FDE)* Client worried about cost? Show `usage` logging, cap `max_tokens`, default to Haiku, trim resent context.


## Assignment
- **Beginner:** change the first call's prompt and print only `content[0].text`.
- **Intermediate:** extend `ask_claude` to also print total tokens and estimated cost.
- **Advanced:** add a simple two-turn conversation by resending the previous user+assistant messages, and observe how `input_tokens` grows.
- **Project:** extend the Support Reply Drafter to read complaints from a list of dicts (with a customer name) and personalize each reply, then print the batch's estimated cost.


## Assessment

**Multiple choice (10)**
1. An API call to Claude is: (a) stateful (b) stateless (c) cached (d) random
2. The answer text is at: (a) reply.text (b) reply.message (c) reply.content[0].text (d) reply.answer
3. `max_tokens` limits the: (a) prompt (b) reply (c) model (d) key
4. `stop_reason == "max_tokens"` means: (a) error (b) answer was cut off (c) success (d) tool needed
5. You are billed for: (a) input only (b) output only (c) input and output (d) neither
6. Best default model for learning: (a) opus (b) sonnet (c) haiku (d) none
7. `temperature=0.0` gives answers that are: (a) creative (b) focused/repeatable (c) longer (d) cheaper
8. `system` is used to: (a) store the key (b) set rules/persona (c) count tokens (d) pick a model
9. Conversation memory lives in: (a) the API (b) your code (resent messages) (c) the key (d) the model
10. Where should the API key live in production? (a) the browser (b) your backend (c) the prompt (d) the URL

**Short answer (5)**
1. Why is resending history expensive?
2. Name the three required fields of a basic call.
3. What does `reply.usage` contain and why care?
4. Give one cost-reduction technique.
5. What's the difference between Claude.ai and the Claude API?

**Scenario (3)**
1. Answers come back cut off mid-sentence. Diagnose and fix.
2. Compliance needs identical wording every run. What settings?
3. Monthly bill tripled with no new users. What do you log to find out why?


## Answer Key

**MCQ:** 1-b, 2-c, 3-b, 4-b, 5-c, 6-c, 7-b, 8-b, 9-b, 10-b

**Short answer:**
1. Each turn you resend prior messages; those are input tokens you pay for again, so cost grows with conversation length.
2. `model`, `max_tokens`, `messages`.
3. `input_tokens` and `output_tokens` — they determine your bill and let you track/forecast cost.
4. Any of: shorter prompts, smaller `max_tokens`, trim resent history, use a smaller model (Haiku).
5. Claude.ai is the chat app for humans; the API is a pay-per-token doorway for your code — same models, different access.

**Scenario:**
1. `stop_reason` is `"max_tokens"` — raise `max_tokens` or ask for shorter output.
2. `temperature=0.0`, a strict `system` prompt fixing format, and a pinned model ID.
3. Log per-call `input_tokens`/`output_tokens` (and prompt size) — likely a long resent history or large `max_tokens` driving output up.
